# Week 8, Day 3: Structured picks and a phone alert

**Lesson:** [week8/day3.ipynb](../../../week8/day3.ipynb), *Scanner agent and messenger agent*. **About 40 minutes.**

**What you practice here**

- Writing pydantic models for Structured Outputs, whose field descriptions reach the model inside the JSON schema and work as instructions.
- Building the selection prompts from a list of raw listings: a system message with the rules, then a user message with every listing.
- Getting typed results with the Chat Completions API's `parse` method and a `response_format`, and reading the parsed answer.
- Sending a push notification to your phone with Pushover's API, through `requests.post`.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import os, pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT is None and ("google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ):
    # in Colab: fetch the checker from your fork on GitHub, and the packages Colab lacks
    import subprocess, urllib.request
    ROOT = pathlib.Path("/content/fork")
    (ROOT / "plan").mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/starlest/llm_engineering/main/plan/check.py", ROOT / "plan" / "check.py")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "python-dotenv", "tiktoken"], check=False)
    print("In Colab, the checks run here too. To keep your work, use File > Save a copy in GitHub, to this notebook's path in your fork.")
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week8/day3", globals())
else:  # outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here.")

## The task

The lesson's scanner reads deals from RSS feeds and asks `gpt-5-mini` to pick the five with the most detailed
description and a clear price. It asks with Structured Outputs, passing a pydantic class as the response format, so
the answer comes back as pydantic objects, not as text. Then a push notification reaches your phone through Pushover.
Build the same for a marketplace of wrestling merchandise: your own pydantic models, the prompts, the parse call, and
the push.

`pick_listings` takes the client as its first parameter, and `push` takes `requests.post`; both call them as the
lesson does. The checks hand them stand-ins shaped like the real ones: the client records what you send and answers
with JSON that fits your models, and the post records what it would send. No model is called, and nothing reaches
your phone.

## What each call should give

```python
Listing.model_json_schema()["properties"]["price"]
# {'description': 'The price the seller asks, as a number; never an amount off', 'title': 'Price', 'type': 'number'}

messages_for(LISTINGS, k=3)
# [{'role': 'system', 'content': '... the most detailed description ... a clear price ... $40 off is not a price ...'},
#  {'role': 'user', 'content': 'Pick the 3 listings ...\n\nTitle: Undertaker 1991 Hasbro figure with urn\nDetails: ...'}]

picks = pick_listings(client, LISTINGS)     # one parse request with your Shortlist as the response format
picks.listings[0]
# Listing(summary='A 1991 Hasbro figure of the Undertaker, complete with its urn ...', price=45.5,
#         url='https://merch.example/l/101', wrestler='The Undertaker')

push(requests.post, "The Undertaker, $45.50: https://merch.example/l/101")   # <Response [200]>
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.
import os

from pydantic import BaseModel, Field

### 1 · `class Listing(BaseModel)`

A pydantic model for one listing that the model picks: four required fields, each with a description of your own.
The fields are `summary` (a string), `price` (a float), `url` (a string) and `wrestler` (a string), and each gets a
field description of at least 15 characters. With Structured Outputs these descriptions reach the model inside the
JSON schema, so they work as instructions, as the course's `Deal` shows. Say that `summary` is a short paragraph about
the item itself, `price` the price asked (never an amount off), `url` the listing's URL as given, and `wrestler` the
wrestler, team or event that the item is about, or `"none"`.

<details><summary>Hint</summary>

```python
summary: str = Field(description="...")
```

</details>

In [ ]:
class Listing(BaseModel):
    # Write the four fields here, as described above.
    pass

In [ ]:
check("Listing")

### 2 · `class Shortlist(BaseModel)`

A pydantic model for the model's whole answer: one required field, `listings`, a list of `Listing` objects. Give it a
field description of at least 15 characters that says which listings to pick, as the course's `DealSelection` does
for its `deals`.

<details><summary>Hint</summary>

```python
listings: list[Listing] = Field(description="...")
```

</details>

In [ ]:
class Shortlist(BaseModel):
    # Write the field here, as described above.
    pass

In [ ]:
check("Shortlist")

### 3 · `messages_for(listings, k=3)`

Builds the system and user messages that ask for the best `k` listings. `listings` are raw marketplace listings: dicts
with `"title"`, `"details"` and `"url"`. Return a system message, then a user message. The system message says that
the model picks the listings with the most detailed description and a clear price, and warns that "$40 off" is not a
price, as the lesson's does. The user message asks for exactly `k` listings, writing the number, then gives each
listing's title, details and URL, in the order given and separated by blank lines, as the lesson's
`make_user_prompt` does with `describe()`.

<details><summary>Hint</summary>

```python
"\n\n".join(f"Title: {item['title']}\nDetails: {item['details']}\nURL: {item['url']}" for item in listings)
```

</details>

In [ ]:
def messages_for(listings, k=3):
    # Write messages_for here, as described above.
    raise NotImplementedError("messages_for is not written yet")

In [ ]:
check("messages_for")

### 4 · `pick_listings(client, listings, k=3, model="gpt-5-mini")`

Asks the model for the best `k` listings with Structured Outputs, and returns them as a `Shortlist`. Make one call to
the Chat Completions API's parse method (not create), with the model `model`, the messages from
`messages_for(listings, k)`, your `Shortlist` class itself as the response format, and the reasoning effort
`"minimal"`, as the lesson does. Take the parsed `Shortlist` from the reply's message. Remove the listings whose price
is 0 or less, keep the others in order, and return the `Shortlist`.

<details><summary>Hint</summary>

```python
response = client.chat.completions.parse(model=model, messages=..., response_format=Shortlist, reasoning_effort="minimal")
picks = response.choices[0].message.parsed
```

</details>

In [ ]:
def pick_listings(client, listings, k=3, model="gpt-5-mini"):
    # Write pick_listings here, as described above.
    raise NotImplementedError("pick_listings is not written yet")

In [ ]:
check("pick_listings")

### 5 · `push(post, message)`

Sends a push notification to your phone through Pushover. `post` is `requests.post`. Each time `push` is called, read
`PUSHOVER_USER` and `PUSHOVER_TOKEN` from the environment. Then post once to Pushover's messages URL,
`https://api.pushover.net/1/messages.json`, with the form data `"user"`, `"token"` and `"message"`, as the lesson's
`push` does; extra Pushover fields, such as a `"sound"`, are fine. Return what `post` returns. If either variable is
missing or empty, raise `ValueError` and post nothing.

<details><summary>Hint</summary>

```python
user = os.getenv("PUSHOVER_USER")
return post(url, data={"user": user, "token": token, "message": message})
```

</details>

In [ ]:
def push(post, message):
    # Write push here, as described above.
    raise NotImplementedError("push is not written yet")

In [ ]:
check("push")

## Check everything

The cell below runs all 8 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 8, Day 3 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. Write the steps in the cell at the end; they may call a model with your key, and the checks leave them out.

Each call to `gpt-5-mini` costs well under a cent, even with the 30 deals of the real feed. The push needs the
Pushover app on your phone and its two keys in your `.env`, as the lesson shows.

1. Load your keys with `load_dotenv(override=True)`, make `client = OpenAI()`, and pick from the `LISTINGS` in the
   cell below with `pick_listings(client, LISTINGS)`. Print each pick's price, wrestler, summary and URL.
2. Pick from the real deals too. Put the course's `week8` folder on the path with
   `sys.path.insert(0, str(ROOT / "week8"))`, fetch the deals with `ScrapedDeal.fetch(show_progress=True)` from
   `agents.deals`, turn each into `{"title": d.title, "details": d.details, "url": d.url}`, and call
   `pick_listings(client, feed, k=5)`. See what `wrestler` says for electronics.
3. Push your best merch pick from step 1 to your phone with `push(requests.post, ...)`.

<details><summary>Stuck? One way to write it</summary>

```python
import sys

import requests
from dotenv import load_dotenv
from openai import OpenAI

LISTINGS = [
    {"title": "Undertaker 1991 Hasbro figure with urn",
     "details": "Complete, light wear on the hat, original urn included. $45.50, ships free.",
     "url": "https://merch.example/l/101"},
    {"title": "Winged Eagle replica title belt",
     "details": "Adult size, real leather strap, zinc plates. $40 off this week!",
     "url": "https://merch.example/l/102"},
    {"title": "WrestleMania III program",
     "details": "1987, Pontiac Silverdome, a few creases on the cover. $60.",
     "url": "https://merch.example/l/103"},
    {"title": "Macho Man sunglasses",
     "details": "Pink wraparound shades, a replica of the ring pair. Make an offer.",
     "url": "https://merch.example/l/104"},
    {"title": "Hulkamania sticker sheet",
     "details": "Free with any order over $50.",
     "url": "https://merch.example/l/105"},
    {"title": "Bret Hart Hasbro figure, 1992",
     "details": "Pink and black gear, sunglasses included, card unopened. $38.",
     "url": "https://merch.example/l/106"},
]

load_dotenv(override=True)
client = OpenAI()
picks = pick_listings(client, LISTINGS)
for pick in picks.listings:
    print(f"${pick.price:.2f}  {pick.wrestler}: {pick.summary[:70]}  {pick.url}")
```

```python
sys.path.insert(0, str(ROOT / "week8"))  # the course's week8 folder
from agents.deals import ScrapedDeal

feed = [{"title": d.title, "details": d.details, "url": d.url} for d in ScrapedDeal.fetch(show_progress=True)]
for pick in pick_listings(client, feed, k=5).listings:
    print(f"${pick.price:.2f}  {pick.wrestler}: {pick.summary[:70]}")
```

```python
best = picks.listings[0]
push(requests.post, f"{best.wrestler}: {best.summary[:120]} for ${best.price:.2f} {best.url}")
```

</details>

In [ ]:
# Try it for real: write the steps above here, in as many cells as you like. This cell may call a model with your key; the checks leave it out.
# Raw listings from a wrestling-merchandise marketplace:
LISTINGS = [
    {"title": "Undertaker 1991 Hasbro figure with urn",
     "details": "Complete, light wear on the hat, original urn included. $45.50, ships free.",
     "url": "https://merch.example/l/101"},
    {"title": "Winged Eagle replica title belt",
     "details": "Adult size, real leather strap, zinc plates. $40 off this week!",
     "url": "https://merch.example/l/102"},
    {"title": "WrestleMania III program",
     "details": "1987, Pontiac Silverdome, a few creases on the cover. $60.",
     "url": "https://merch.example/l/103"},
    {"title": "Macho Man sunglasses",
     "details": "Pink wraparound shades, a replica of the ring pair. Make an offer.",
     "url": "https://merch.example/l/104"},
    {"title": "Hulkamania sticker sheet",
     "details": "Free with any order over $50.",
     "url": "https://merch.example/l/105"},
    {"title": "Bret Hart Hasbro figure, 1992",
     "details": "Pink and black gear, sunglasses included, card unopened. $38.",
     "url": "https://merch.example/l/106"},
]